# IB9LQ0 – Generative AI and AI Applications
## Required Task 10 – Visual Risk Scorecard for Property Underwriting

**Student Name:** Jashwanth Anand Shankar  
**Student Number:** *(add your student number here)*  
**Module:** IB9LQ0 | Warwick Business School | 2025–2026

---

### Key Insights and Takeaways

**Task Overview**

This task extends the MLM9 CNN damage classifier beyond statistical accuracy and converts its predictions into a business-oriented risk scorecard for property underwriting. Instead of simply reporting whether the model predicted the right class, we compute an `expected_damage_score` (a probability-weighted average over four damage classes) and map it to three underwriting actions: Auto-clear, Human review, and Urgent inspection / reserve adjustment.

**Key Insight 1 – Softmax probabilities carry more information than a hard class label**

A model that predicts minor-damage with 51% confidence and one that predicts it with 99% confidence both produce the same class label, but they should trigger very different business responses. The `expected_damage_score` captures this uncertainty — a property with probability mass spread across minor and major damage will score higher than one with near-certain no-damage, even if both are labelled the same class.

**Key Insight 2 – Business thresholds are a design choice, not a statistical one**

The cutoffs (0.75 and 1.75) determine how sensitive the pipeline is. The evaluation metrics — particularly the recall of Urgent inspection for major/destroyed properties — directly quantify this trade-off and allow an actuary to adjust the thresholds to match the organisation's risk appetite.

**Key Insight 3 – Business utility metrics matter more than accuracy alone**

A model with 70% accuracy might still be highly valuable if it correctly routes 95% of the severe-damage properties to urgent inspection. The scorecard evaluation is therefore structured around recall for high-severity classes and precision for the Auto-clear bucket.

**Personal Takeaway**

Building the scorecard reinforced that deploying an ML model in a business context requires translating model outputs into decision-relevant quantities. The MLM9 CNN gives us logits; the softmax, weighted score, and threshold rules convert those logits into an actionable, auditable recommendation — the difference between a research prototype and a production underwriting tool.

---
*AI Disclosure: Claude (Anthropic) was used to assist in structuring and drafting this code. All outputs, interpretations, and conclusions were reviewed and validated by the student.*

## Data setup
Upload `xbd_teaching_subset.zip` to the Colab runtime before running. Expected contents after extraction:

```
xbd_teaching_subset/
  labels.csv
  pre_images/
  post_images/
```

This cell sets up the data by checking for the zipped dataset, extracting it if necessary, and then loading the `labels.csv` file into a pandas DataFrame. It also displays basic information about the loaded labels.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
ZIP_PATH = Path('xbd_teaching_subset.zip')
DATA_DIR = Path('xbd_teaching_subset')
if not ZIP_PATH.exists():
    matches = list(Path.cwd().rglob('xbd_teaching_subset.zip'))
    if matches:
        ZIP_PATH = matches[0]
if not DATA_DIR.exists():
    assert ZIP_PATH.exists(), 'Upload xbd_teaching_subset.zip to the Colab runtime, then rerun this cell.'
    print('Extracting:', ZIP_PATH.resolve())
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('.')
else:
    print('Using existing extracted folder:', DATA_DIR.resolve())
if not (DATA_DIR/'labels.csv').exists() and Path('labels.csv').exists():
    DATA_DIR = Path('.')
labels = pd.read_csv(DATA_DIR/'labels.csv')
print('DATA_DIR:', DATA_DIR.resolve())
print('Rows:', len(labels))
display(labels.head())
display(labels.damage_label.value_counts())
display(pd.crosstab(labels.split, labels.damage_label))

This cell imports essential libraries for data manipulation, image processing, deep learning with PyTorch, and plotting. It also defines global constants like the device (CPU/GPU), sets a random seed for reproducibility, defines mappings for damage labels, and includes utility functions for loading images and displaying image grids.

In [ ]:
import math, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(42)
DAMAGE_TO_ID={'no-damage':0,'minor-damage':1,'major-damage':2,'destroyed':3}
ID_TO_DAMAGE={v:k for k,v in DAMAGE_TO_ID.items()}
def load_image(path, size=64):
    img=Image.open(path).convert('RGB').resize((size,size))
    arr=np.array(img).astype('float32')/255.0
    arr=arr*2.0-1.0
    return torch.from_numpy(arr).permute(2,0,1)
def show_grid(xs, titles=None, ncols=4, figsize=(10,6)):
    xs=xs.detach().cpu().clamp(-1,1); xs=(xs+1)/2
    n=len(xs); ncols=min(ncols,n); nrows=math.ceil(n/ncols)
    plt.figure(figsize=figsize)
    for i in range(n):
        plt.subplot(nrows,ncols,i+1); plt.imshow(xs[i].permute(1,2,0).numpy()); plt.axis('off')
        if titles is not None: plt.title(titles[i], fontsize=8)
    plt.tight_layout(); plt.show()

## Dataset

This cell defines the `PostDamageDataset` class, a custom PyTorch Dataset for loading images and their corresponding damage labels. It then initializes the training, validation, and test datasets and dataloaders, and displays a sample of images from the training set with their labels.

In [ ]:
class PostDamageDataset(Dataset):
    def __init__(self,data_dir,labels,split='train',image_size=64):
        self.data_dir=Path(data_dir); self.df=labels[labels['split'].eq(split)].reset_index(drop=True); self.image_size=image_size
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        r=self.df.iloc[idx]; return load_image(self.data_dir/r.post_image,self.image_size), int(r.damage_id)
IMAGE_SIZE=64; BATCH_SIZE=64
train_ds=PostDamageDataset(DATA_DIR,labels,'train',IMAGE_SIZE); val_ds=PostDamageDataset(DATA_DIR,labels,'val',IMAGE_SIZE); test_ds=PostDamageDataset(DATA_DIR,labels,'test',IMAGE_SIZE)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0); val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0); test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
print('Train:',len(train_ds),'Val:',len(val_ds),'Test:',len(test_ds))
x,y=next(iter(train_loader)); show_grid(x[:8],[ID_TO_DAMAGE[int(c)] for c in y[:8]],ncols=4,figsize=(10,5))

## Class balance and weights

This cell analyzes the class balance within the training data and calculates class weights. These weights are used in the loss function to mitigate the impact of class imbalance during model training.

In [ ]:
display(pd.crosstab(labels['split'],labels['damage_label']))
class_counts=labels[labels['split'].eq('train')]['damage_id'].value_counts().sort_index(); class_weights=torch.ones(4)
for cls_id in range(4):
    if cls_id in class_counts.index: class_weights[cls_id]=len(train_ds)/(4*class_counts.loc[cls_id])
class_weights=class_weights.to(DEVICE); print('Class weights:',class_weights.detach().cpu().numpy())

## Model

This cell defines the `DamageCNN` class, which is a compact Convolutional Neural Network (CNN) for image classification. It specifies the architecture of the neural network and then initializes an instance of the model and sets up the AdamW optimizer for training.

In [ ]:
class DamageCNN(nn.Module):
    def __init__(self,num_classes=4):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(128,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(128,num_classes))
    def forward(self,x): return self.net(x)
model=DamageCNN().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)

## Train and evaluate

This cell defines the `evaluate` function to assess the model's performance on a given dataset by calculating loss, accuracy, and predictions. It also defines `per_class_accuracy` to show performance for each damage category. The main training loop then iterates for 50 epochs, performing forward and backward passes, updating weights, and evaluating the model on the validation set after each epoch. Finally, it plots the training history.

In [ ]:
def evaluate(model,loader):
    model.eval(); total=correct=0; loss_sum=0; ys=[]; ps=[]
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(DEVICE),y.to(DEVICE); logits=model(x); loss=F.cross_entropy(logits,y,weight=class_weights); pred=logits.argmax(1)
            total+=y.numel(); correct+=(pred==y).sum().item(); loss_sum+=loss.item()*y.numel(); ys.append(y.cpu()); ps.append(pred.cpu())
    return {'loss':loss_sum/max(total,1),'accuracy':correct/max(total,1),'y_true':torch.cat(ys).numpy() if ys else np.array([]),'y_pred':torch.cat(ps).numpy() if ps else np.array([])}
def per_class_accuracy(y_true,y_pred):
    return pd.DataFrame([{'damage_class':ID_TO_DAMAGE[i],'n':int((y_true==i).sum()),'accuracy':np.nan if (y_true==i).sum()==0 else float((y_pred[y_true==i]==i).mean())} for i in range(4)])
EPOCHS=50; history=[]
for epoch in range(1,EPOCHS+1):
    model.train(); losses=[]
    for x,y in train_loader:
        x,y=x.to(DEVICE),y.to(DEVICE); loss=F.cross_entropy(model(x),y,weight=class_weights)
        opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
    val=evaluate(model,val_loader); history.append({'epoch':epoch,'train_loss':float(np.mean(losses)),'val_loss':val['loss'],'val_accuracy':val['accuracy']})
    print(f"Epoch {epoch:03d} | train_loss={np.mean(losses):.4f} | val_loss={val['loss']:.4f} | val_acc={val['accuracy']:.3f}")
hist=pd.DataFrame(history); display(hist)
plt.figure(figsize=(6,4)); plt.plot(hist.epoch,hist.train_loss,label='train loss'); plt.plot(hist.epoch,hist.val_loss,label='validation loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('Training curve'); plt.show()
plt.figure(figsize=(6,4)); plt.plot(hist.epoch,hist.val_accuracy,label='validation accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.ylim(0,1); plt.legend(); plt.title('Validation accuracy'); plt.show()

## Test-set evaluation

This cell evaluates the trained model on the unseen test set, reporting the overall test loss and accuracy. It also displays a detailed per-class accuracy breakdown and a confusion matrix to visualize the model's performance across different damage categories.

In [ ]:
test=evaluate(model,test_loader); print('Test loss:',round(test['loss'],4)); print('Test accuracy:',round(test['accuracy'],3))
y_true=test['y_true']; y_pred=test['y_pred']; display(per_class_accuracy(y_true,y_pred))
cm=pd.crosstab(pd.Series(y_true).map(ID_TO_DAMAGE),pd.Series(y_pred).map(ID_TO_DAMAGE),rownames=['Actual'],colnames=['Predicted'],dropna=False)
class_names=[ID_TO_DAMAGE[i] for i in range(4)]; cm=cm.reindex(index=class_names,columns=class_names,fill_value=0); display(cm)
plt.figure(figsize=(6,5)); plt.imshow(cm.values); plt.xticks(range(4),cm.columns,rotation=45,ha='right'); plt.yticks(range(4),cm.index); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Damage Classification Confusion Matrix')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]): plt.text(j,i,int(cm.values[i,j]),ha='center',va='center')
plt.tight_layout(); plt.show()

#Required Task 10
##Visual Risk Scorecard for Property Underwriting

## Step 1: Load the Test Data

Select only the records where `split == "test"` from `labels` and store them in `df_test_properties`, exactly as specified in the task.

In [ ]:
df_test_properties = labels[labels["split"].eq("test")].copy()
df_test_properties = df_test_properties.reset_index(drop=True)

print(f"Test properties loaded: {len(df_test_properties)} records")
print("Damage label distribution in test set:")
display(df_test_properties['damage_label'].value_counts())
display(df_test_properties[['post_image', 'damage_label', 'damage_id']].head())

## Step 3: Generate Predicted Class Probabilities

We iterate through the test DataLoader, run each batch through the trained model, apply `F.softmax` to convert logits to probabilities, and collect `p_no_damage`, `p_minor_damage`, `p_major_damage`, `p_destroyed`, and `predicted_damage_label` for every property.

In [ ]:
all_p_no_damage    = []
all_p_minor_damage = []
all_p_major_damage = []
all_p_destroyed    = []
all_pred_labels    = []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        logits = model(x)
        probs  = F.softmax(logits, dim=1).cpu().numpy()
        pred_ids = probs.argmax(axis=1)
        all_p_no_damage.extend(probs[:, 0].tolist())
        all_p_minor_damage.extend(probs[:, 1].tolist())
        all_p_major_damage.extend(probs[:, 2].tolist())
        all_p_destroyed.extend(probs[:, 3].tolist())
        all_pred_labels.extend([ID_TO_DAMAGE[int(i)] for i in pred_ids])

df_test_properties['p_no_damage']            = all_p_no_damage
df_test_properties['p_minor_damage']         = all_p_minor_damage
df_test_properties['p_major_damage']         = all_p_major_damage
df_test_properties['p_destroyed']            = all_p_destroyed
df_test_properties['predicted_damage_label'] = all_pred_labels

print("Probabilities and predicted labels added to df_test_properties.")
display(df_test_properties[
    ['post_image', 'damage_label', 'predicted_damage_label',
     'p_no_damage', 'p_minor_damage', 'p_major_damage', 'p_destroyed']
].head(8))

## Step 4: Compute an Expected Damage Score and Assign an Action

The expected damage score is the probability-weighted average of the damage severity scale (0–3):

`expected_damage_score = 0*p_no_damage + 1*p_minor_damage + 2*p_major_damage + 3*p_destroyed`

Each score is then mapped to a recommended underwriting action using the thresholds from the task.

In [ ]:
df_test_properties['expected_damage_score'] = (
    0 * df_test_properties['p_no_damage']    +
    1 * df_test_properties['p_minor_damage'] +
    2 * df_test_properties['p_major_damage'] +
    3 * df_test_properties['p_destroyed']
)

print("expected_damage_score summary:")
print(df_test_properties['expected_damage_score'].describe().round(4))

In [ ]:
def assign_action(score):
    if score < 0.75:
        return 'Auto-clear'
    elif score < 1.75:
        return 'Human review'
    else:
        return 'Urgent inspection / reserve adjustment'

df_test_properties['recommended_action'] = (
    df_test_properties['expected_damage_score'].apply(assign_action)
)

print("recommended_action distribution:")
display(df_test_properties['recommended_action'].value_counts())

## Step 5: Evaluate the Risk Scorecard

The evaluation goes beyond simple accuracy to assess business utility across six metrics.

In [ ]:
# Overall classification accuracy
correct_mask = (
    df_test_properties['predicted_damage_label'] == df_test_properties['damage_label']
)
overall_accuracy = correct_mask.mean()
print(f"Overall classification accuracy: {overall_accuracy:.4f} "
      f"({correct_mask.sum()} / {len(df_test_properties)})")

In [ ]:
# Confusion matrix — same pd.crosstab + imshow pattern as MLM9
cm_scorecard = pd.crosstab(
    df_test_properties['damage_label'],
    df_test_properties['predicted_damage_label'],
    rownames=['Actual'],
    colnames=['Predicted'],
    dropna=False
)
class_names = [ID_TO_DAMAGE[i] for i in range(4)]
cm_scorecard = cm_scorecard.reindex(index=class_names, columns=class_names, fill_value=0)
display(cm_scorecard)
plt.figure(figsize=(6,5)); plt.imshow(cm_scorecard.values)
plt.xticks(range(4),cm_scorecard.columns,rotation=45,ha='right'); plt.yticks(range(4),cm_scorecard.index)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Scorecard — Damage Classification Confusion Matrix')
for i in range(cm_scorecard.shape[0]):
    for j in range(cm_scorecard.shape[1]): plt.text(j,i,int(cm_scorecard.values[i,j]),ha='center',va='center')
plt.tight_layout(); plt.show()

In [ ]:
# Average expected_damage_score by true damage_label
avg_score_by_class = (
    df_test_properties
    .groupby('damage_label')['expected_damage_score']
    .mean()
    .reindex(class_names)
    .round(4)
)
print("Average expected_damage_score by true damage_label:")
display(avg_score_by_class)

In [ ]:
# Number of properties assigned to each recommended_action
action_counts = df_test_properties['recommended_action'].value_counts().reset_index()
action_counts.columns = ['recommended_action', 'number_of_properties']
action_counts['percentage_of_test_set'] = (
    action_counts['number_of_properties'] / len(df_test_properties) * 100
).round(2)
action_order = ['Auto-clear', 'Human review', 'Urgent inspection / reserve adjustment']
action_counts = (
    action_counts.set_index('recommended_action')
    .reindex(action_order, fill_value=0)
    .reset_index()
)
print("Properties assigned to each recommended action:")
display(action_counts)

In [ ]:
# % of truly major-damage or destroyed routed to Urgent inspection
severe_mask  = df_test_properties['damage_label'].isin(['major-damage', 'destroyed'])
urgent_mask  = df_test_properties['recommended_action'] == 'Urgent inspection / reserve adjustment'
n_severe         = severe_mask.sum()
n_severe_caught  = (severe_mask & urgent_mask).sum()
pct_severe_caught = (n_severe_caught / n_severe * 100) if n_severe > 0 else 0.0
print(f"Truly major-damage or destroyed properties       : {n_severe}")
print(f"Routed to 'Urgent inspection / reserve adj.'     : {n_severe_caught}")
print(f"  -> {pct_severe_caught:.1f}% of high-severity properties correctly flagged")
print()
# % of truly no-damage routed to Auto-clear
no_damage_mask      = df_test_properties['damage_label'] == 'no-damage'
auto_clear_mask     = df_test_properties['recommended_action'] == 'Auto-clear'
n_no_damage         = no_damage_mask.sum()
n_no_damage_cleared = (no_damage_mask & auto_clear_mask).sum()
pct_no_damage_cleared = (n_no_damage_cleared / n_no_damage * 100) if n_no_damage > 0 else 0.0
print(f"Truly no-damage properties                       : {n_no_damage}")
print(f"Routed to 'Auto-clear'                           : {n_no_damage_cleared}")
print(f"  -> {pct_no_damage_cleared:.1f}% of no-damage properties correctly auto-cleared")

## Step 6: Display Results

Display a sample of `df_test_properties` with the columns specified in the task, and the summary action table.

In [ ]:
# Sample of df_test_properties with task-specified columns
display_cols = [
    'post_image', 'damage_label', 'predicted_damage_label',
    'p_no_damage', 'p_minor_damage', 'p_major_damage', 'p_destroyed',
    'expected_damage_score', 'recommended_action'
]
df_display_sample = df_test_properties[display_cols].copy()
for col in ['p_no_damage','p_minor_damage','p_major_damage','p_destroyed','expected_damage_score']:
    df_display_sample[col] = df_display_sample[col].round(4)
print(f"Sample of df_test_properties ({len(df_test_properties)} total rows, showing first 10):")
display(df_display_sample.head(10))

In [ ]:
# Summary table — recommended_action, number_of_properties, percentage_of_test_set
print("Summary table: Recommended Action Distribution")
display(action_counts)

## Step 7: Short Written Interpretation

### Does the scorecard separate low-risk and high-risk properties?

The average `expected_damage_score` by true damage class tells the core story. If the scores increase monotonically from `no-damage` (near 0) through `minor-damage`, `major-damage`, to `destroyed` (near 3), the scorecard is doing useful work — it is converting the model's probabilistic output into a risk ladder that a business decision-maker can act on. Even imperfect separation is valuable if the high-severity recall is strong, because the cost of missing a destroyed property far exceeds the cost of over-routing a minor-damage property to human review.

### Which damage classes are most often confused?

From the confusion matrix, the most common misclassifications tend to occur at adjacent severity levels: `minor-damage` predicted as `no-damage` (or vice versa) and `major-damage` confused with `minor-damage` or `destroyed`. This is expected — the visual difference between a slightly and moderately damaged roof can be subtle in a 64×64 satellite patch. Errors at adjacent levels are less dangerous for the scorecard than cross-severity errors because the probability mass usually spills into neighbouring classes, keeping the `expected_damage_score` elevated and triggering a higher-tier action.

### Is the action threshold too conservative or too lenient?

The threshold pair (0.75 / 1.75) is a deliberate design choice. If the high-severity recall (percentage of major/destroyed routed to Urgent) is below approximately 80%, the thresholds are too lenient and should be lowered. If the Auto-clear bucket contains many minor-damage properties, the lower threshold is too low and should be raised. In an underwriting context, erring on the side of caution is typically preferable — the financial and reputational risk of missing a destroyed property exceeds the cost of an additional adjuster visit.

### Would you trust this model for full automation?

Not yet. The Auto-clear bucket is the only action that should even approach automation, and only when the no-damage Auto-clear rate is high (above 90%) and the model's false-negative rate for severe damage into Auto-clear is near zero. A compact CNN trained on 64×64 patches with a class-imbalanced dataset is a strong starting point for triage and prioritisation, but the final underwriting decision — especially for major or destroyed properties — should retain a human in the loop. The scorecard's value lies in routing: drastically reducing the number of properties a human adjuster must review from scratch, not in eliminating the adjuster entirely.